# Séance 6 · Modéliser : ton premier modèle de machine learning · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

Jusqu'ici, c'est toi qui posais les questions aux données. Aujourd'hui, c'est la machine qui **apprend une règle** à partir d'exemples, puis devine sur des cas qu'elle n'a jamais vus. Trois espèces de manchots vivent en Antarctique (Adélie, Chinstrap, Gentoo) ; des scientifiques ont mesuré 344 d'entre eux. À partir des mesures seules, une machine peut-elle deviner l'espèce ?

Ce notebook tourne dans **Google Colab** (rien à installer). Clique sur une cellule et fais `Maj + Entrée` pour l'exécuter.

**Livrable de la séance** : un premier modèle entraîné et évalué, avec une phrase pour expliquer ce qu'il a compris.


## Préparation

**scikit-learn** (abrégé `sklearn`) est la boîte à outils du machine learning en Python. Déjà installée dans Colab.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, mean_absolute_error

URL_MANCHOTS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
try:
    df = pd.read_csv(URL_MANCHOTS)
    print("Chargé :", df.shape[0], "manchots,", df.shape[1], "colonnes")
except Exception as erreur:
    print("Pas de réseau ? Impossible de charger le fichier :", erreur)
df.head()

## 1. L'apprentissage supervisé, expliqué simplement

Quand tu apprends à reconnaître les champignons, on ne te donne pas une règle : on te montre des exemples **avec la réponse** (« celui-ci est comestible, celui-là non »). À force, tu trouves la règle toi-même. C'est exactement l'**apprentissage supervisé** : des exemples + la bonne réponse → la machine trouve la règle → elle l'applique à des cas nouveaux.

Le vocabulaire :
- **X** (les *features*) : ce que la machine voit (les mesures du manchot),
- **y** (la *cible*) : ce qu'elle doit deviner (l'espèce),
- **entraîner** (`fit`) : trouver la règle ; **prédire** (`predict`) : l'appliquer.

Deux familles de problèmes :

| Question | Réponse attendue | Famille |
|---|---|---|
| Ce mail est-il un spam ? | oui / non | **Classification** (une catégorie) |
| Combien de vues aura cette vidéo ? | 12 400 | **Régression** (un nombre) |
| Quelle espèce est ce manchot ? | Adélie / Chinstrap / Gentoo | Classification |
| Combien pèse ce manchot ? | 4 250 g | Régression |

Quatre colonnes de mesures forment X, la colonne réponse forme y.

*Quatre colonnes de mesures forment X, la colonne réponse forme y.*

In [ ]:
# Un mini-exemple à la main : 6 mails, 2 indices (nombre de "!!!" et de liens), la réponse (spam ?)
mails = pd.DataFrame({"nb_exclamations": [0, 5, 1, 7, 0, 4],
                      "nb_liens":        [1, 6, 0, 3, 2, 5],
                      "spam":            ["non", "oui", "non", "oui", "non", "oui"]})
X_mails = mails[["nb_exclamations", "nb_liens"]]      # ce que la machine voit
y_mails = mails["spam"]                               # ce qu'elle doit deviner

modele_mails = DecisionTreeClassifier(max_depth=1).fit(X_mails, y_mails)   # apprend UNE question
nouveau_mail = pd.DataFrame({"nb_exclamations": [3], "nb_liens": [4]})
colonne_choisie = X_mails.columns[modele_mails.tree_.feature[0]]     # la question que l'arbre a trouvée
print(f"La règle apprise : spam si {colonne_choisie} > {modele_mails.tree_.threshold[0]:.1f}")
print("Nouveau mail avec 3 '!' et 4 liens →", modele_mails.predict(nouveau_mail)[0])

**Exercice** : classification ou régression ? Pour chaque cas, écris `"C"` ou `"R"` dans le dictionnaire, puis exécute pour vérifier ton score.

<details><summary>Solution</summary>

```python
reponses = {"note_film": "R", "genre_musique": "C", "temps_trajet": "R", "photo_chat_ou_chien": "C", "prix_appart": "R", "langue_du_texte": "C"}
```
</details>

In [ ]:
# À toi : remplace les "?" par "C" (classification) ou "R" (régression)
reponses = {
    "note_film": "?",              # deviner la note qu'un film aura sur 10
    "genre_musique": "?",          # deviner le genre d'une chanson (rap, pop, rock...)
    "temps_trajet": "?",           # deviner la durée d'un trajet en minutes
    "photo_chat_ou_chien": "?",    # deviner si une photo montre un chat ou un chien
    "prix_appart": "?",            # deviner le prix d'un appartement
    "langue_du_texte": "?",        # deviner la langue d'un texte
}
corrige = {"note_film": "R", "genre_musique": "C", "temps_trajet": "R", "photo_chat_ou_chien": "C", "prix_appart": "R", "langue_du_texte": "C"}
score = sum(reponses[k] == corrige[k] for k in corrige)
print(f"Score : {score} / 6", "🎉" if score == 6 else "(les '?' comptent faux, réessaie)")

## 2. Explorer et nettoyer avant de modéliser

Réflexe de la séance 4 et 5 : on regarde avant de calculer. Combien de manchots par espèce ? Des cases vides ? Les mesures séparent-elles déjà les espèces à l'œil nu ?

In [ ]:
print(df.isna().sum())
print()
print(df["species"].value_counts())
# Un modèle n'aime pas les cases vides : on retire les lignes incomplètes (il y en a peu)
df = df.dropna().reset_index(drop=True)
print("\nIl reste", len(df), "manchots après nettoyage")
df.groupby("species")[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]].mean().round(1)

In [ ]:
couleurs = {"Adelie": "tab:blue", "Chinstrap": "tab:orange", "Gentoo": "tab:green"}
plt.figure(figsize=(7, 5))
for espece, groupe in df.groupby("species"):
    plt.scatter(groupe["bill_length_mm"], groupe["flipper_length_mm"], label=espece, alpha=0.7, c=couleurs[espece])
plt.xlabel("Longueur du bec (mm)"); plt.ylabel("Longueur de la nageoire (mm)")
plt.legend(); plt.title("On voit déjà trois groupes... la machine aussi ?")
plt.show()

**Exercice** : refais le nuage de points avec `bill_depth_mm` et `body_mass_g`. Les groupes sont-ils aussi bien séparés ? Quelle espèce se détache le plus ?

<details><summary>Solution</summary>

```python
plt.figure(figsize=(7, 5))
for espece, groupe in df.groupby("species"):
    plt.scatter(groupe["bill_depth_mm"], groupe["body_mass_g"], label=espece, alpha=0.7, c=couleurs[espece])
plt.xlabel("Épaisseur du bec (mm)"); plt.ylabel("Poids (g)"); plt.legend(); plt.show()
# Gentoo se détache (bec fin, gros poids) ; Adelie et Chinstrap se mélangent
```
</details>

In [ ]:
# À toi

## 3. Entraînement / test : pourquoi on cache une partie des données

Si un prof te donne les réponses de l'examen la veille, ta note ne prouve rien. Pareil pour un modèle : pour savoir s'il a **compris** (et pas juste **appris par cœur**), on l'évalue sur des données qu'il n'a **jamais vues**.

Donc on coupe le tableau en deux : **train** (75 %, pour apprendre) et **test** (25 %, caché, pour vérifier). `random_state=42` fixe le hasard pour que tout le monde ait la même coupe.

On coupe en deux avant d'entraîner : la partie cachée ne sert qu'à vérifier.

*On coupe en deux avant d'entraîner : la partie cachée ne sert qu'à vérifier.*

In [ ]:
colonnes = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = df[colonnes]          # ce que la machine voit
y = df["species"]         # ce qu'elle doit deviner

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print("Apprentissage :", len(X_train), "manchots · Test (cachés) :", len(X_test))

La **précision** (*accuracy*) = le pourcentage de bonnes réponses sur les données de test. Avant tout modèle, calcule la précision d'un **modèle bête** qui répond toujours l'espèce la plus fréquente : c'est la barre à battre.

In [ ]:
espece_frequente = y_train.value_counts().index[0]
precision_bete = (y_test == espece_frequente).mean()
print(f"Modèle bête (répond toujours « {espece_frequente} ») : {precision_bete * 100:.1f} % de bonnes réponses")

**Exercice** : recoupe avec `test_size=0.5` (la moitié cachée). Combien de manchots restent pour apprendre ? Puis change `random_state` (12, 7...) : la précision du modèle bête bouge-t-elle ? Pourquoi ?

<details><summary>Solution</summary>

```python
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.5, random_state=12)
print("Apprentissage :", len(Xa), "· Test :", len(Xb))
print("Modèle bête :", round((yb == ya.value_counts().index[0]).mean() * 100, 1), "%")
# La précision bouge un peu : le hasard de la coupe change quels manchots sont cachés.
# On garde la coupe 75 / 25 avec random_state=42 pour la suite.
```
</details>

In [ ]:
# À toi

## 4. Premier modèle : l'arbre de décision

Un **arbre de décision** pose des questions en cascade, comme au jeu « Qui est-ce ? » : « la nageoire fait-elle plus de 206 mm ? » → oui : Gentoo ; non : « le bec fait-il plus de 43 mm ? »... On peut le **lire**, c'est parfait pour comprendre ce que la machine a compris.

In [ ]:
arbre = DecisionTreeClassifier(max_depth=3, random_state=42)
arbre.fit(X_train, y_train)                        # 1. apprendre sur les exemples

predictions = arbre.predict(X_test)                # 2. deviner sur les cachés
precision = accuracy_score(y_test, predictions)    # 3. mesurer
print(f"Précision sur les données de test : {precision * 100:.1f} %   (modèle bête : {precision_bete * 100:.1f} %)")

plt.figure(figsize=(16, 8))
plot_tree(arbre, feature_names=colonnes, class_names=arbre.classes_, filled=True, fontsize=9)
plt.title("L'arbre lu de haut en bas : chaque case pose une question, les feuilles donnent l'espèce")
plt.show()

Comment lire une case : la **question** en haut (`flipper_length_mm <= 206.5`), `samples` = le nombre de manchots arrivés là, `value` = combien de chaque espèce, `class` = l'espèce majoritaire. À gauche si la réponse est « oui », à droite si « non ».

Où se trompe-t-il ? Un tableau croisé « vraie espèce × espèce prédite » le montre : la diagonale, ce sont les bonnes réponses.

In [ ]:
pd.crosstab(y_test, predictions, rownames=["Vraie espèce"], colnames=["Prédite"])

**Exercice** : invente un manchot (4 mesures) et fais-le deviner à l'arbre. Puis suis son chemin dans l'arbre à la main pour vérifier que tu obtiens la même espèce.

<details><summary>Solution</summary>

```python
nouveau = pd.DataFrame([[45.0, 15.0, 215.0, 5000.0]], columns=colonnes)
print("Espèce prédite :", arbre.predict(nouveau)[0])
# nageoire 215 > 206.5 → à droite → Gentoo
```
</details>

In [ ]:
# À toi

## 5. Un deuxième modèle et le sur-apprentissage

Les **k plus proches voisins** (k-NN) : pour deviner un manchot, on regarde les *k* manchots qui lui ressemblent le plus dans les exemples, et on prend l'espèce majoritaire. Pas de règle apprise, juste de la mémoire et de la ressemblance.

In [ ]:
voisins = KNeighborsClassifier(n_neighbors=5)
voisins.fit(X_train, y_train)
print(f"Précision k-NN (k=5) : {accuracy_score(y_test, voisins.predict(X_test)) * 100:.1f} %")
print(f"Précision arbre      : {precision * 100:.1f} %")

Et si on laissait l'arbre poser autant de questions qu'il veut ? Il finit par apprendre **par cœur** chaque manchot du train (100 % de précision dessus)... et se trompe davantage sur les cachés. C'est le **sur-apprentissage** (*overfitting*) : comme réviser en apprenant les corrigés par cœur au lieu de comprendre. On le voit en traçant la précision train et test selon la profondeur.

In [ ]:
profondeurs = range(1, 21)
prec_train, prec_test = [], []
for p in profondeurs:
    a = DecisionTreeClassifier(max_depth=p, random_state=42).fit(X_train, y_train)
    prec_train.append(accuracy_score(y_train, a.predict(X_train)))
    prec_test.append(accuracy_score(y_test, a.predict(X_test)))

plt.figure(figsize=(8, 4.5))
plt.plot(profondeurs, prec_train, marker="o", label="sur les exemples (train)")
plt.plot(profondeurs, prec_test, marker="s", label="sur les cachés (test)")
plt.xlabel("Profondeur de l'arbre (max_depth)"); plt.ylabel("Précision"); plt.xticks(profondeurs)
plt.legend(); plt.title("Sur-apprentissage : le train monte à 100 %, le test stagne ou baisse")
plt.show()

**Exercice** : fais la même courbe pour k-NN avec `n_neighbors` de 1 à 30. Que se passe-t-il avec k = 1 (précision train) ? Et avec un très grand k ?

<details><summary>Solution</summary>

```python
ks = range(1, 31)
pt, pv = [], []
for k in ks:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    pt.append(accuracy_score(y_train, m.predict(X_train))); pv.append(accuracy_score(y_test, m.predict(X_test)))
plt.plot(ks, pt, marker="o", label="train"); plt.plot(ks, pv, marker="s", label="test")
plt.xlabel("k"); plt.ylabel("Précision"); plt.legend(); plt.show()
# k=1 : 100 % en train (chaque manchot est son propre voisin) ; k très grand : le modèle répond presque toujours l'espèce majoritaire
```
</details>

In [ ]:
# À toi

## 6. Une régression rapide : prédire le poids

Changeons de famille : deviner un **nombre**, le poids du manchot (`body_mass_g`), à partir de ses autres mesures. La **régression linéaire** cherche la meilleure formule du type `poids = a × nageoire + b × bec + ...`.

On ne parle plus de précision (« bonne ou mauvaise réponse » n'a pas de sens pour un nombre) mais d'**erreur moyenne** : de combien de grammes on se trompe en moyenne.

In [ ]:
colonnes_poids = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]
Xp_train, Xp_test, yp_train, yp_test = train_test_split(df[colonnes_poids], df["body_mass_g"], test_size=0.25, random_state=42)

regression = LinearRegression().fit(Xp_train, yp_train)
poids_predits = regression.predict(Xp_test)
erreur = mean_absolute_error(yp_test, poids_predits)
print(f"Erreur moyenne : {erreur:.0f} g (sur des manchots de {yp_test.min():.0f} à {yp_test.max():.0f} g)")
print("La formule apprise : poids =", " + ".join(f"{c:.1f} × {n}" for c, n in zip(regression.coef_, colonnes_poids)), f"+ {regression.intercept_:.0f}")

plt.figure(figsize=(5.5, 5.5))
plt.scatter(yp_test, poids_predits, alpha=0.6)
plt.plot([2500, 6500], [2500, 6500], "r--", label="prédiction parfaite")
plt.xlabel("Vrai poids (g)"); plt.ylabel("Poids prédit (g)"); plt.legend()
plt.title("Plus les points sont près de la ligne rouge, mieux c'est")
plt.show()

**Exercice** : que vaut l'erreur d'un « modèle bête » qui prédit toujours le poids moyen du train ? Compare avec la régression. Puis essaie la régression avec la seule colonne `flipper_length_mm` : perd-on beaucoup ?

<details><summary>Solution</summary>

```python
poids_moyen = yp_train.mean()
print("Modèle bête :", round(mean_absolute_error(yp_test, [poids_moyen] * len(yp_test))), "g")
r1 = LinearRegression().fit(Xp_train[["flipper_length_mm"]], yp_train)
print("Nageoire seule :", round(mean_absolute_error(yp_test, r1.predict(Xp_test[["flipper_length_mm"]]))), "g")
```
</details>

In [ ]:
# À toi

## 7. On mesure, on améliore

Un data scientist ne s'arrête pas au premier score. La boucle du métier : **mesurer → changer une chose → re-mesurer**. Trois leviers simples :

1. **Ajouter des informations** : le sexe et l'île du manchot sont dans le tableau, mais ce sont des mots. `pd.get_dummies` les transforme en colonnes 0/1 que le modèle comprend.
2. **Régler le modèle** : `max_depth`, `n_neighbors` (on l'a vu).
3. **Changer de modèle** : arbre, voisins, et plus tard forêts, réseaux de neurones...

In [ ]:
# Levier 1 : ajouter sexe et île, transformés en colonnes 0/1
X_plus = pd.get_dummies(df[colonnes + ["sex", "island"]], dtype=int)
print("Nouvelles colonnes :", list(X_plus.columns))
Xa_train, Xa_test, ya_train, ya_test = train_test_split(X_plus, y, test_size=0.25, random_state=42)

resultats = {}
for nom, modele in [("arbre depth=3", DecisionTreeClassifier(max_depth=3, random_state=42)),
                    ("arbre depth=5", DecisionTreeClassifier(max_depth=5, random_state=42)),
                    ("k-NN k=5", KNeighborsClassifier(n_neighbors=5))]:
    avant = accuracy_score(y_test, modele.fit(X_train, y_train).predict(X_test))
    apres = accuracy_score(ya_test, modele.fit(Xa_train, ya_train).predict(Xa_test))
    resultats[nom] = {"4 mesures": round(avant * 100, 1), "+ sexe et île": round(apres * 100, 1)}
pd.DataFrame(resultats).T

**Exercice** : k-NN calcule des distances, et le poids (en milliers de grammes) écrase les autres mesures (en dizaines de mm). Mets toutes les colonnes à la même échelle avec `StandardScaler` (`from sklearn.preprocessing import StandardScaler`) et re-mesure k-NN. Ça change ?

<details><summary>Solution</summary>

```python
from sklearn.preprocessing import StandardScaler
echelle = StandardScaler().fit(X_train)
knn = KNeighborsClassifier(n_neighbors=5).fit(echelle.transform(X_train), y_train)
print("k-NN avec mise à l'échelle :", round(accuracy_score(y_test, knn.predict(echelle.transform(X_test))) * 100, 1), "%")
# souvent proche de 100 % : la mise à l'échelle est indispensable pour k-NN
```
</details>

In [ ]:
# À toi

## 8. Projet (80 min) : la même recette sur les Pokémon

À toi de refaire toute la recette sur les 800 Pokémon. La cible principale est **Legendary** (option A) ; les options B et C sont là si tu finis en avance :

- **Option A (classification, 2 classes)** : prédire `Legendary` (le Pokémon est-il légendaire ?) à partir de ses stats. Attention : seulement 65 légendaires sur 800, donc le **modèle bête** (« jamais légendaire ») a déjà 92 % de précision. Regarde le tableau croisé : combien de légendaires ton modèle retrouve-t-il vraiment ?
- **Option B (classification, 3 classes)** : prédire `Type 1` parmi 3 types seulement (`Water`, `Fire`, `Grass`) à partir des stats. Plus dur : les stats suffisent-elles à deviner le type ?
- **Option C (régression)** : prédire `Total` à partir de `HP`, `Attack`, `Defense`, `Speed`... trop facile ? Essaie alors de prédire `Speed` à partir des autres stats.

La recette (une étape par cellule) :
1. Charger, explorer (`value_counts`, un nuage de points coloré par la cible)
2. Choisir X et y, couper train / test
3. Le modèle bête : la barre à battre
4. Un arbre de décision : précision (ou erreur moyenne), et **lire l'arbre** (`max_depth=3` pour qu'il reste lisible)
5. Un deuxième modèle ou un réglage : mesurer, comparer
6. **Le livrable** : le meilleur score, et **une phrase** qui explique ce que le modèle a compris (« un Pokémon avec plus de 580 de Total et une Sp. Atk élevée est presque toujours légendaire »)

In [ ]:
# Question 1 : charger et explorer
URL_POKEMON = "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv"
try:
    pokemon = pd.read_csv(URL_POKEMON)
except Exception as erreur:
    print("Pas de réseau ?", erreur)
stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
print(pokemon["Legendary"].value_counts())
pokemon.head(3)

### Le piège du « toujours non »

Avant de te lancer, mesure le modèle le plus bête possible : il répond « pas légendaire » à tout le monde. Son score a l'air excellent... et pourtant il ne trouve **aucun** légendaire. Sur des classes **déséquilibrées** (92 % / 8 %), la précision seule ment : regarde toujours le tableau croisé, et compte combien de légendaires ton modèle retrouve vraiment.

In [ ]:
toujours_non = pd.Series(False, index=pokemon.index)              # prédit « pas légendaire » pour tous
print(f"Précision du « toujours non » : {accuracy_score(pokemon['Legendary'], toujours_non) * 100:.1f} %")
print(pd.crosstab(pokemon["Legendary"], toujours_non, rownames=["Vrai"], colnames=["Prédit"]))
print("→ 0 légendaire retrouvé sur", pokemon["Legendary"].sum(), ". Un bon score, un modèle inutile. La barre à battre, c'est ça.")

In [ ]:
# Question 2 : X (les stats), y (Legendary), train / test (ajoute stratify=y pour garder 8 % de légendaires des deux côtés)

In [ ]:
# Question 3 : le modèle bête sur TES données de test (la barre à battre)

In [ ]:
# Question 4 : arbre de décision, score, plot_tree, tableau croisé (combien de légendaires retrouvés ?)

In [ ]:
# Question 5 : un deuxième modèle ou un réglage, comparer

In [ ]:
# Question 6 : le livrable (à remplir)
meilleur_modele = "..."
score = "..."
phrase = "Mon modèle a compris que ..."
print(f"Modèle : {meilleur_modele} · Score : {score}\n{phrase}")

<details><summary>Solution (option A, Legendary)</summary>

```python
X = pokemon[stats]; y = pokemon["Legendary"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Modèle bête (jamais légendaire) :", round((y_test == False).mean() * 100, 1), "%")

arbre = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
pred = arbre.predict(X_test)
print("Arbre :", round(accuracy_score(y_test, pred) * 100, 1), "%")     # 92 %... comme le modèle bête !
print(pd.crosstab(y_test, pred, rownames=["Vrai"], colnames=["Prédit"]))  # mais il retrouve 3 légendaires sur 16
plt.figure(figsize=(16, 7)); plot_tree(arbre, feature_names=stats, class_names=["Normal", "Légendaire"], filled=True, fontsize=9); plt.show()

# Réglage : un arbre plus profond retrouve 9 légendaires sur 16 pour 92.5 % (la précision bouge à peine, le tableau croisé, lui, change)
arbre5 = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_train, y_train)
print(pd.crosstab(y_test, arbre5.predict(X_test), rownames=["Vrai"], colnames=["Prédit"]))
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
print("k-NN :", round(accuracy_score(y_test, knn.predict(X_test)) * 100, 1), "%")   # 93.5 %, 7 légendaires retrouvés

phrase = "Mon modèle a compris qu'un Pokémon avec une attaque spéciale au-dessus de 147, rapide et avec une grosse attaque, est presque toujours légendaire."
```
</details>

<details><summary>Solution (option B, 3 types)</summary>

```python
trois = pokemon[pokemon["Type 1"].isin(["Water", "Fire", "Grass"])]
X = trois[stats]; y = trois["Type 1"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print("Modèle bête :", round((y_test == y_train.value_counts().index[0]).mean() * 100, 1), "%")
arbre = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
print("Arbre :", round(accuracy_score(y_test, arbre.predict(X_test)) * 100, 1), "%")
# Les stats ne disent presque rien du type : un bon modèle ne peut pas deviner ce qui n'est pas dans les données !
```
</details>

## À retenir

- **Apprentissage supervisé** : des exemples avec la réponse → la machine trouve la règle → elle prédit sur du nouveau
- **Classification** = deviner une catégorie ; **régression** = deviner un nombre
- **X** = ce que la machine voit, **y** = ce qu'elle doit deviner ; `fit` apprend, `predict` devine
- **Train / test** : on cache 25 % des données pour vérifier que le modèle a compris, pas appris par cœur
- **Précision** = % de bonnes réponses ; **erreur moyenne** = de combien on se trompe ; toujours comparer au **modèle bête**
- Un arbre trop profond fait du **sur-apprentissage** : 100 % en train, moins bien en test
- Le métier : **mesurer, changer une chose, re-mesurer**

## Pour montrer aux autres

Pendant les 20 dernières minutes, chacun présente son modèle Pokémon. Trois questions guides :

1. Quel est ton score, et celui du modèle bête ? Ton modèle sert-il vraiment à quelque chose ?
2. Ta phrase : qu'a compris ton modèle ? (lis-la dans l'arbre)
3. Qu'as-tu changé pour améliorer le score, et de combien ?

## Liens gratuits

- Le dataset des manchots expliqué : https://allisonhorst.github.io/palmerpenguins/
- scikit-learn, guide de démarrage : https://scikit-learn.org/stable/getting_started.html
- Une visualisation animée du machine learning : http://www.r2d3.us/une-introduction-visuelle-au-machine-learning-1/
- Kaggle Learn, cours « Intro to Machine Learning » (gratuit) : https://www.kaggle.com/learn/intro-to-machine-learning